# 03 — LIBERO single-frame RGB-D observation probe

本 Notebook 只验证一帧 LIBERO 观测：RGB、MuJoCo depth buffer、米制 depth、相机内外参和末端位姿。它不调用 3D HAMSTER、不训练策略、不修改 SmolVLA。

所有依赖安装、导入和运行都显式使用 `/content/.venv/bin/python`。Planner 的 `/content/.venv_planner` 不会被修改。首次运行会下载固定 revision 的 `lerobot/libero-assets`。

In [33]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("VLA_PROJECT_ROOT", "/content/vla-3d-trajectory-guidance")).resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "libero_probe_colab.json"
assert CONFIG_PATH.is_file(), f"missing project config: {CONFIG_PATH}"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
CORE_CONFIG = json.loads((PROJECT_ROOT / CONFIG["protected_core_config"]).read_text(encoding="utf-8"))
LIBERO_PY = Path(CONFIG["libero"]["env_dir"]) / "bin" / "python"

def run(command, *, env=None):
    command = list(map(str, command))
    print("$", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    returncode = process.wait()
    output = "".join(lines)
    if returncode:
        raise subprocess.CalledProcessError(returncode, command, output=output)
    return subprocess.CompletedProcess(command, returncode, stdout=output)

print(f"Project: {PROJECT_ROOT}")
print(f"Target interpreter: {LIBERO_PY}")
print(f"Protected Python: {CORE_CONFIG['python_version']}")

Project: /content/vla-3d-trajectory-guidance
Target interpreter: /content/.venv/bin/python
Protected Python: 3.12.13


## 1. GPU、驱动与 uv 检查

这里不做 Planner 推理，只确认服务器和 EGL 所需的 GPU 可见。`uv` 必须保持参考版本 0.13.6；若当前运行时没有 `uv`，先运行 01 Notebook 中的 uv 安装单元。

In [34]:
run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free", "--format=csv,noheader"])
uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("uv not found; run the uv==0.13.6 setup cell from 01_planner_setup.ipynb first")
uv_version = subprocess.check_output([uv, "--version"], text=True).strip()
assert CORE_CONFIG["uv_version"] in uv_version, (uv_version, CORE_CONFIG["uv_version"])
print(uv_version)

$ nvidia-smi --query-gpu=name,driver_version,memory.total,memory.free --format=csv,noheader
NVIDIA RTX PRO 6000 Blackwell Server Edition, 580.82.07, 97887 MiB, 97251 MiB
uv 0.13.6 (x86_64-unknown-linux-gnu)


## 2. 创建并固定 `/content/.venv`

若环境不存在，先只用 uv 创建 Python 3.12.13 环境；随后所有 Python 安装逻辑都由目标解释器执行。脚本安装固定 LeRobot commit 的 `[libero]` extra，不安装 SmolVLA 训练组件。

In [35]:
if not LIBERO_PY.is_file():
    run([uv, "python", "install", CORE_CONFIG["python_version"]])
    run([uv, "venv", "--managed-python", "--python", CORE_CONFIG["python_version"], CONFIG["libero"]["env_dir"]])
assert LIBERO_PY.is_file(), LIBERO_PY
run([
    LIBERO_PY,
    PROJECT_ROOT / "scripts" / "bootstrap_libero_env.py",
    "--project-root", PROJECT_ROOT,
    "--config", CONFIG_PATH,
])

$ /content/.venv/bin/python /content/vla-3d-trajectory-guidance/scripts/bootstrap_libero_env.py --project-root /content/vla-3d-trajectory-guidance --config /content/vla-3d-trajectory-guidance/configs/libero_probe_colab.json

$ /usr/local/bin/uv python install 3.12.13
Python 3.12.13 is already installed
Verified target interpreter: /content/.venv/bin/python (Python 3.12.13)

$ /usr/local/bin/uv pip install --python /content/.venv/bin/python --torch-backend cu128 --constraint /content/vla-3d-trajectory-guidance/configs/libero_constraints.txt torch==2.11.0+cu128 torchvision==0.26.0+cu128 torchcodec==0.11.0+cu128 numpy==2.0.2 nvidia-npp-cu12==12.4.1.87
Checked 5 packages in 2ms

$ /usr/local/bin/uv pip install --python /content/.venv/bin/python --torch-backend cu128 --constraint /content/vla-3d-trajectory-guidance/configs/libero_constraints.txt https://files.pythonhosted.org/packages/2a/79/c286b894c051988d062241682834df915c945bcf51009ffdffbe5ecf69bf/hf_libero-0.1.4-py3-none-any.whl#sha256=

CompletedProcess(args=['/content/.venv/bin/python', '/content/vla-3d-trajectory-guidance/scripts/bootstrap_libero_env.py', '--project-root', '/content/vla-3d-trajectory-guidance', '--config', '/content/vla-3d-trajectory-guidance/configs/libero_probe_colab.json'], returncode=0, stdout='\n$ /usr/local/bin/uv python install 3.12.13\nPython 3.12.13 is already installed\nVerified target interpreter: /content/.venv/bin/python (Python 3.12.13)\n\n$ /usr/local/bin/uv pip install --python /content/.venv/bin/python --torch-backend cu128 --constraint /content/vla-3d-trajectory-guidance/configs/libero_constraints.txt torch==2.11.0+cu128 torchvision==0.26.0+cu128 torchcodec==0.11.0+cu128 numpy==2.0.2 nvidia-npp-cu12==12.4.1.87\nChecked 5 packages in 2ms\n\n$ /usr/local/bin/uv pip install --python /content/.venv/bin/python --torch-backend cu128 --constraint /content/vla-3d-trajectory-guidance/configs/libero_constraints.txt https://files.pythonhosted.org/packages/2a/79/c286b894c051988d062241682834df9

## 3. 精确版本、CUDA 与导入检查

Planner 环境使用 Transformers 4.57.6；这里的独立环境使用固定 LeRobot 所需的 Transformers 5.5.4。两者不会混装。

In [36]:
probe_env = os.environ.copy()
probe_env["MUJOCO_GL"] = CONFIG["libero"]["render_backend"]
probe_env["MPLBACKEND"] = CONFIG["libero"]["matplotlib_backend"]
probe_env["LIBERO_CONFIG_PATH"] = CONFIG["libero"]["config_dir"]
python_major_minor = ".".join(CORE_CONFIG["python_version"].split(".")[:2])
npp_dir = (
    Path(CONFIG["libero"]["env_dir"])
    / "lib"
    / f"python{python_major_minor}"
    / "site-packages"
    / "nvidia"
    / "npp"
    / "lib"
)
npp_libraries = sorted(npp_dir.glob("libnppicc.so*"))
assert npp_libraries, f"NVIDIA NPP library not found under {npp_dir}"
current_loader_path = probe_env.get("LD_LIBRARY_PATH", "")
loader_paths = [path for path in current_loader_path.split(os.pathsep) if path]
if str(npp_dir) not in loader_paths:
    probe_env["LD_LIBRARY_PATH"] = str(npp_dir) + (
        os.pathsep + current_loader_path if current_loader_path else ""
    )
print(f"NPP loader path: {npp_dir}")
run([
    LIBERO_PY,
    PROJECT_ROOT / "scripts" / "check_libero_environment.py",
    "--project-root", PROJECT_ROOT,
    "--config", CONFIG_PATH,
    "--gpu-test",
], env=probe_env)

NPP loader path: /content/.venv/lib/python3.12/site-packages/nvidia/npp/lib
$ /content/.venv/bin/python /content/vla-3d-trajectory-guidance/scripts/check_libero_environment.py --project-root /content/vla-3d-trajectory-guidance --config /content/vla-3d-trajectory-guidance/configs/libero_probe_colab.json --gpu-test
[1/6] Checking target interpreter and render backend...
[2/6] Checking protected distribution versions...
{
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "torchcodec": "0.11.0+cu128",
  "numpy": "2.0.2",
  "nvidia-npp-cu12": "12.4.1.87",
  "transformers": "5.5.4",
  "huggingface-hub": "1.33.0",
  "hf-libero": "0.1.4",
  "robosuite": "1.4.0",
  "robomimic": "0.2.0",
  "mujoco": "3.8.1",
  "lerobot": "0.6.2"
}
[3/6] Checking the pinned LeRobot checkout...
[4/6] Importing MuJoCo, robosuite, LIBERO, and protected ML packages...
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init_

CompletedProcess(args=['/content/.venv/bin/python', '/content/vla-3d-trajectory-guidance/scripts/check_libero_environment.py', '--project-root', '/content/vla-3d-trajectory-guidance', '--config', '/content/vla-3d-trajectory-guidance/configs/libero_probe_colab.json', '--gpu-test'], returncode=0, stdout='[1/6] Checking target interpreter and render backend...\n[2/6] Checking protected distribution versions...\n{\n  "torch": "2.11.0+cu128",\n  "torchvision": "0.26.0+cu128",\n  "torchcodec": "0.11.0+cu128",\n  "numpy": "2.0.2",\n  "nvidia-npp-cu12": "12.4.1.87",\n  "transformers": "5.5.4",\n  "huggingface-hub": "1.33.0",\n  "hf-libero": "0.1.4",\n  "robosuite": "1.4.0",\n  "robomimic": "0.2.0",\n  "mujoco": "3.8.1",\n  "lerobot": "0.6.2"\n}\n[3/6] Checking the pinned LeRobot checkout...\n[4/6] Importing MuJoCo, robosuite, LIBERO, and protected ML packages...\n[robosuite WARNING] No private macro file found! (__init__.py:7)\n[robosuite WARNING] It is recommended to use a private macro file 

## 4. 采集单帧 RGB-D 与标定

探测器会把 robosuite 图像约定显式设置为 OpenCV（左上角原点、u 向右、v 向下），用官方 `get_real_depth_map` 把非线性 depth buffer 转为米，并同时保存 `T_world_camera` 与 `T_camera_world`。

In [37]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
probe_cfg = CONFIG["probe"]
OUTPUT_DIR = Path(probe_cfg["output_root"]) / (
    f"{probe_cfg['suite']}_task{probe_cfg['task_id']:02d}_"
    f"init{probe_cfg['init_state_id']:03d}_{run_id}"
)
run([
    LIBERO_PY,
    PROJECT_ROOT / "scripts" / "run_libero_observation_probe.py",
    "--project-root", PROJECT_ROOT,
    "--config", CONFIG_PATH,
    "--output-dir", OUTPUT_DIR,
], env=probe_env)
print(f"Probe output: {OUTPUT_DIR}")

$ /content/.venv/bin/python /content/vla-3d-trajectory-guidance/scripts/run_libero_observation_probe.py --project-root /content/vla-3d-trajectory-guidance --config /content/vla-3d-trajectory-guidance/configs/libero_probe_colab.json --output-dir /content/vla-3d-trajectory-guidance/outputs/libero_observation_probe/libero_spatial_task00_init000_20261101T104011Z

Fetching 586 files: 100%|██████████| 586/586 [00:26<00:00, 22.20it/s]
[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /content/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Traceback (most recent call last):
  File "/content/vla-3d-trajectory-guidance/scripts/run_libero_observation_probe.py", line 417, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/content/vla-3d-traje

CalledProcessError: Command '['/content/.venv/bin/python', '/content/vla-3d-trajectory-guidance/scripts/run_libero_observation_probe.py', '--project-root', '/content/vla-3d-trajectory-guidance', '--config', '/content/vla-3d-trajectory-guidance/configs/libero_probe_colab.json', '--output-dir', '/content/vla-3d-trajectory-guidance/outputs/libero_observation_probe/libero_spatial_task00_init000_20261101T104011Z']' returned non-zero exit status 1.

## 5. 查看结果并执行验收门

RGB 和 depth preview 只用于目视检查；后续几何计算必须读取 `*_depth_m.npy` 与 `*_calibration.json`，不能从预览图恢复数值。

In [ ]:
from IPython.display import Image as DisplayImage, display

manifest_path = OUTPUT_DIR / "observation_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({
    "valid": manifest["valid"],
    "task": manifest["task"],
    "coordinate_conventions": manifest["coordinate_conventions"],
    "timing_seconds": manifest["timing_seconds"],
}, ensure_ascii=False, indent=2))
for camera_name, result in manifest["cameras"].items():
    print(f"\n{camera_name}: valid={result['valid']}")
    display(DisplayImage(filename=result["rgb"]["path"]))
    display(DisplayImage(filename=result["artifacts"]["depth_preview_png"]))

In [ ]:
assert manifest["valid"] is True, json.dumps(manifest["cameras"], ensure_ascii=False, indent=2)
assert manifest["provenance"]["lerobot_repo_commit"] == CONFIG["libero"]["repo_commit"]
assert manifest["provenance"]["assets_revision"] == CONFIG["libero"]["assets_revision"]
assert manifest["coordinate_conventions"]["metric_depth"] == "camera_z_depth_m"
for camera_name, result in manifest["cameras"].items():
    assert result["valid"], camera_name
    assert result["pointcloud_sample_count"] > 0, camera_name
    for artifact in result["artifacts"].values():
        assert Path(artifact).is_file(), artifact
print("LIBERO single-frame observation probe passed. No planner or policy was run.")